# Importar librerías necesarias
Este notebook importa las librerías principales utilizadas en el proyecto para el procesamiento y análisis de datos.

In [1]:
# Importación de librerías principales para el proyecto
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import streamlit as st
import holidays

# Cargar archivo de ventas_2025_inferencia
En esta sección se carga el archivo 'ventas_2025_inferencia.csv' en un DataFrame llamado inferencia_df para su análisis.

In [2]:
# Cargar el archivo de ventas_2025_inferencia.csv en un DataFrame
inferencia_path = '../data/raw/inferencia/ventas_2025_inferencia.csv'
inferencia_df = pd.read_csv(inferencia_path)

# Mostrar las primeras filas para verificar la carga
display(inferencia_df.head())

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [6]:
# Transformación completa de inferencia_df para forecasting

import pandas as pd
import numpy as np
import holidays

# Cargar inferencia_df
inferencia_path = '../data/raw/inferencia/ventas_2025_inferencia.csv'
inferencia_df = pd.read_csv(inferencia_path)

# 1. Conversión de fecha y variables temporales
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.dayofweek
inferencia_df['dia_semana_es'] = inferencia_df['fecha'].dt.day_name(locale='es_ES')
inferencia_df['nombre_dia_semana'] = inferencia_df['fecha'].dt.day_name(locale='es_ES')

# 2. Festivos, fines de semana, laborables, Black Friday, Cyber Monday, semana/día año, primer/último día mes, trimestre, quincena, víspera/post festivo
festivos_es = holidays.country_holidays('ES', years=inferencia_df['año'].unique())
inferencia_df['es_fin_de_semana'] = inferencia_df['dia_semana'].isin([5, 6])
inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(festivos_es)
inferencia_df['es_laborable'] = (~inferencia_df['es_fin_de_semana']) & (~inferencia_df['es_festivo'])
inferencia_df['semana_año'] = inferencia_df['fecha'].dt.isocalendar().week
inferencia_df['dia_año'] = inferencia_df['fecha'].dt.dayofyear
inferencia_df['es_primer_dia_mes'] = inferencia_df['dia_mes'] == 1
inferencia_df['es_ultimo_dia_mes'] = inferencia_df['fecha'] == (inferencia_df['fecha'] + pd.offsets.MonthEnd(0))
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['es_primer_quincena'] = inferencia_df['dia_mes'] <= 15
inferencia_df['es_víspera_festivo'] = (inferencia_df['fecha'] + pd.Timedelta(days=1)).isin(festivos_es)
inferencia_df['es_post_festivo'] = (inferencia_df['fecha'] - pd.Timedelta(days=1)).isin(festivos_es)

# Black Friday y Cyber Monday
def es_black_friday(fecha):
    if fecha.month == 11 and fecha.weekday() == 4:
        ultimo_viernes = max([d for d in pd.date_range(start=f'{fecha.year}-11-01', end=f'{fecha.year}-11-30') if d.weekday() == 4])
        return fecha == ultimo_viernes
    return False

def es_cyber_monday(fecha):
    if fecha.month == 11 and fecha.weekday() == 0:
        ultimo_viernes = max([d for d in pd.date_range(start=f'{fecha.year}-11-01', end=f'{fecha.year}-11-30') if d.weekday() == 4])
        cyber_monday = ultimo_viernes + pd.Timedelta(days=3)
        return fecha == cyber_monday
    return False

inferencia_df['es_black_friday'] = inferencia_df['fecha'].apply(es_black_friday)
inferencia_df['es_cyber_monday'] = inferencia_df['fecha'].apply(es_cyber_monday)

# 3. Lags y media móvil de 7 días por año
for lag in range(1, 8):
    inferencia_df[f'unidades_vendidas_lag{lag}'] = inferencia_df.groupby('año')['unidades_vendidas'].shift(lag)
inferencia_df['unidades_vendidas_mm7'] = inferencia_df.groupby('año')['unidades_vendidas'].transform(lambda x: x.rolling(window=7, min_periods=7).mean())

# 4. Unión de competencia_df y cálculo de precio_competencia y ratio_precio
competencia_path = '../data/raw/entrenamiento/competencia.csv'
competencia_df = pd.read_csv(competencia_path)
competencia_df['fecha'] = pd.to_datetime(competencia_df['fecha'])
4
3
3

C:\Users\Usuario\AppData\Local\Temp\ipykernel_18900\3956961309.py:23: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(festivos_es)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_18900\3956961309.py:31: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  inferencia_df['es_víspera_festivo'] = (inferencia_df['fecha'] + pd.Timedelta(days=1)).isin(festivos_es)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_18900\3956961309.py:32: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is depre

3

In [7]:
inferencia_df.shape

(888, 40)

In [8]:
inferencia_df.producto_id.nunique() 

24

In [9]:
inferencia_df.fecha.nunique()

37

In [10]:
inferencia_df.fecha.unique()    

<DatetimeArray>
['2025-10-25 00:00:00', '2025-10-26 00:00:00', '2025-10-27 00:00:00',
 '2025-10-28 00:00:00', '2025-10-29 00:00:00', '2025-10-30 00:00:00',
 '2025-10-31 00:00:00', '2025-11-01 00:00:00', '2025-11-02 00:00:00',
 '2025-11-03 00:00:00', '2025-11-04 00:00:00', '2025-11-05 00:00:00',
 '2025-11-06 00:00:00', '2025-11-07 00:00:00', '2025-11-08 00:00:00',
 '2025-11-09 00:00:00', '2025-11-10 00:00:00', '2025-11-11 00:00:00',
 '2025-11-12 00:00:00', '2025-11-13 00:00:00', '2025-11-14 00:00:00',
 '2025-11-15 00:00:00', '2025-11-16 00:00:00', '2025-11-17 00:00:00',
 '2025-11-18 00:00:00', '2025-11-19 00:00:00', '2025-11-20 00:00:00',
 '2025-11-21 00:00:00', '2025-11-22 00:00:00', '2025-11-23 00:00:00',
 '2025-11-24 00:00:00', '2025-11-25 00:00:00', '2025-11-26 00:00:00',
 '2025-11-27 00:00:00', '2025-11-28 00:00:00', '2025-11-29 00:00:00',
 '2025-11-30 00:00:00']
Length: 37, dtype: datetime64[ns]

In [11]:
# Guardar el dataframe transformado en la carpeta data/processed
inferencia_df.to_csv('../data/processed/inferencia_df_transformado.csv', index=False)
print('DataFrame guardado como inferencia_df_transformado.csv en data/processed')

DataFrame guardado como inferencia_df_transformado.csv en data/processed


In [12]:
# Resumen final del dataframe transformado
print('--- Resumen inferencia_df ---')
print(f"Total de registros: {inferencia_df.shape[0]}")
print(f"Total de columnas: {inferencia_df.shape[1]}")
print(f"Productos únicos: {inferencia_df['producto_id'].nunique()}")
print(f"Rango de fechas: {inferencia_df['fecha'].min().date()} a {inferencia_df['fecha'].max().date()}")

--- Resumen inferencia_df ---
Total de registros: 888
Total de columnas: 40
Productos únicos: 24
Rango de fechas: 2025-10-25 a 2025-11-30


In [13]:
# Filtrar solo fechas de noviembre 2025
inferencia_df = inferencia_df[(inferencia_df['fecha'] >= '2025-11-01') & (inferencia_df['fecha'] <= '2025-11-30')].reset_index(drop=True)

# Guardar el dataframe filtrado
inferencia_df.to_csv('../data/processed/inferencia_df_transformado.csv', index=False)

# Mostrar resumen final actualizado
print('--- Resumen inferencia_df (noviembre 2025) ---')
print(f"Total de registros: {inferencia_df.shape[0]}")
print(f"Total de columnas: {inferencia_df.shape[1]}")
print(f"Productos únicos: {inferencia_df['producto_id'].nunique()}")
print(f"Rango de fechas: {inferencia_df['fecha'].min().date()} a {inferencia_df['fecha'].max().date()}")

--- Resumen inferencia_df (noviembre 2025) ---
Total de registros: 720
Total de columnas: 40
Productos únicos: 24
Rango de fechas: 2025-11-01 a 2025-11-30


In [ ]:
# Guardar el dataframe filtrado y transformado en la carpeta data/processed
inferencia_df.to_csv('../data/processed/inferencia_df_transformado.csv', index=False)
print('DataFrame guardado como inferencia_df_transformado.csv en data/processed')